<a href="https://colab.research.google.com/github/kosar-am/day-night-road-cyclegan/blob/main/notebooks/03_cyclegan_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day-to-Night Road Scene Translation using CycleGAN

## CycleGAN Model

In this notebook, we will:

- Understand the CycleGAN architecture
- Build the residual block
- Build the generators
- Build the discriminators
- Initialize model weights
- Test the model with sample tensors
- Verify output shapes and parameter counts

# Import Libraries

In [13]:
import torch
import torch.nn as nn
import pickle
from pathlib import Path
from google.colab import drive
from torchvision import transforms
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from PIL import Image
import kagglehub


# Load Saved Image Lists

In [14]:


# Mount Google Drive
drive.mount("/content/drive")

# Directory containing the saved image lists
save_dir = Path(
    "/content/drive/MyDrive/My-Project/CycleGAN_Project-2026/saved"
)

# Load saved image lists
with open(save_dir / "day_train_images.pkl", "rb") as file:
    day_train_images = pickle.load(file)

with open(save_dir / "day_val_images.pkl", "rb") as file:
    day_val_images = pickle.load(file)

with open(save_dir / "night_train_images.pkl", "rb") as file:
    night_train_images = pickle.load(file)

with open(save_dir / "night_val_images.pkl", "rb") as file:
    night_val_images = pickle.load(file)

print(f"Day train images   : {len(day_train_images)}")
print(f"Day validation     : {len(day_val_images)}")
print(f"Night train images : {len(night_train_images)}")
print(f"Night validation   : {len(night_val_images)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Day train images   : 29440
Day validation     : 7360
Night train images : 22422
Night validation   : 5606


# Define Image Transformations

In [15]:

# Define image size
image_size = 256

# Training transformations
train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

# Validation transformations
val_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

# Build Custom Dataset

In [16]:

class DayNightDataset(Dataset):
    def __init__(
        self,
        day_image_paths,
        night_image_paths,
        transform=None,
    ):
        self.day_image_paths = day_image_paths
        self.night_image_paths = night_image_paths
        self.transform = transform

    def __len__(self):
        return max(
            len(self.day_image_paths),
            len(self.night_image_paths),
        )

    def __getitem__(self, index):
        day_path = self.day_image_paths[
            index % len(self.day_image_paths)
        ]

        night_path = self.night_image_paths[
            index % len(self.night_image_paths)
        ]

        day_image = Image.open(day_path).convert("RGB")
        night_image = Image.open(night_path).convert("RGB")

        if self.transform:
            day_image = self.transform(day_image)
            night_image = self.transform(night_image)

        return {
            "day": day_image,
            "night": night_image,
        }

# Create Datasets and DataLoaders

In [17]:


# Download or locate BDD100K in the current Colab environment
dataset_path = Path(
    kagglehub.dataset_download("alvaromalfaro/bdd100k")
) / "bdd100k"

train_images_path = dataset_path / "images" / "100k" / "train"

# Replace old absolute paths with the current Colab path
def update_image_paths(image_paths, new_directory):
    return [
        new_directory / Path(image_path).name
        for image_path in image_paths
    ]

day_train_images = update_image_paths(
    day_train_images,
    train_images_path,
)

day_val_images = update_image_paths(
    day_val_images,
    train_images_path,
)

night_train_images = update_image_paths(
    night_train_images,
    train_images_path,
)

night_val_images = update_image_paths(
    night_val_images,
    train_images_path,
)

print(day_train_images[0])
print(day_train_images[0].exists())

Using Colab cache for faster access to the 'bdd100k' dataset.
/kaggle/input/bdd100k/bdd100k/images/100k/train/0000f77c-6257be58.jpg
True


In [18]:
missing_paths = sum(
    not path.exists()
    for image_list in [
        day_train_images,
        day_val_images,
        night_train_images,
        night_val_images,
    ]
    for path in image_list
)

print(f"Missing image paths: {missing_paths}")

Missing image paths: 0


In [19]:


# Create training and validation datasets
train_dataset = DayNightDataset(
    day_train_images,
    night_train_images,
    transform=train_transform,
)

val_dataset = DayNightDataset(
    day_val_images,
    night_val_images,
    transform=val_transform,
)

# Define batch size
batch_size = 8

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
)

# Display dataset and batch information
print(f"Training samples   : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Training batches   : {len(train_loader)}")
print(f"Validation batches : {len(val_loader)}")

Training samples   : 29440
Validation samples : 7360
Training batches   : 3680
Validation batches : 920


In [20]:
# Get one batch from the training DataLoader
batch = next(iter(train_loader))

day_batch = batch["day"]
night_batch = batch["night"]

print(f"Day batch shape   : {day_batch.shape}")
print(f"Night batch shape : {night_batch.shape}")

Day batch shape   : torch.Size([8, 3, 256, 256])
Night batch shape : torch.Size([8, 3, 256, 256])


# Generator

## Build Residual Block

In [22]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            # First convolution block
            nn.ReflectionPad2d(1),
            nn.Conv2d(
                in_channels=channels,
                out_channels=channels,
                kernel_size=3,
                stride=1,
                padding=0,
                bias=False,
            ),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),

            # Second convolution block
            nn.ReflectionPad2d(1),
            nn.Conv2d(
                in_channels=channels,
                out_channels=channels,
                kernel_size=3,
                stride=1,
                padding=0,
                bias=False,
            ),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)

In [23]:
# Create one Residual Block with 256 channels
residual_block = ResidualBlock(channels=256)

# Create a sample tensor
sample_input = torch.randn(1, 256, 64, 64)

# Forward pass
sample_output = residual_block(sample_input)

print(f"Input shape : {sample_input.shape}")
print(f"Output shape: {sample_output.shape}")

Input shape : torch.Size([1, 256, 64, 64])
Output shape: torch.Size([1, 256, 64, 64])


## Build Encoder

In [24]:
class Encoder(nn.Module):
    def __init__(self, input_channels=3):
        super().__init__()

        self.model = nn.Sequential(
            # Initial convolution
            nn.ReflectionPad2d(3),
            nn.Conv2d(
                in_channels=input_channels,
                out_channels=64,
                kernel_size=7,
                stride=1,
                padding=0,
                bias=False,
            ),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            # First downsampling
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            # Second downsampling
            nn.Conv2d(
                in_channels=128,
                out_channels=256,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.model(x)

In [25]:
#test encoder
encoder = Encoder(input_channels=3)

sample_input = torch.randn(1, 3, 256, 256)
encoded_output = encoder(sample_input)

print(f"Encoder input shape : {sample_input.shape}")
print(f"Encoder output shape: {encoded_output.shape}")

Encoder input shape : torch.Size([1, 3, 256, 256])
Encoder output shape: torch.Size([1, 256, 64, 64])


## Build Decoder

In [26]:
class Decoder(nn.Module):
    def __init__(self, output_channels=3):
        super().__init__()

        self.model = nn.Sequential(
            # First upsampling
            nn.ConvTranspose2d(
                in_channels=256,
                out_channels=128,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                bias=False,
            ),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            # Second upsampling
            nn.ConvTranspose2d(
                in_channels=128,
                out_channels=64,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                bias=False,
            ),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            # Final convolution
            nn.ReflectionPad2d(3),
            nn.Conv2d(
                in_channels=64,
                out_channels=output_channels,
                kernel_size=7,
                stride=1,
                padding=0,
            ),
            nn.Tanh(),
        )

    def forward(self, x):
        return self.model(x)

In [27]:
#test decoder
decoder = Decoder(output_channels=3)

sample_features = torch.randn(1, 256, 64, 64)
decoded_output = decoder(sample_features)

print(f"Decoder input shape : {sample_features.shape}")
print(f"Decoder output shape: {decoded_output.shape}")

Decoder input shape : torch.Size([1, 256, 64, 64])
Decoder output shape: torch.Size([1, 3, 256, 256])


# Build Generator

In [28]:
class Generator(nn.Module):
    def __init__(
        self,
        input_channels=3,
        output_channels=3,
        num_residual_blocks=9,
    ):
        super().__init__()

        self.encoder = Encoder(
            input_channels=input_channels
        )

        self.residual_blocks = nn.Sequential(
            *[
                ResidualBlock(channels=256)
                for _ in range(num_residual_blocks)
            ]
        )

        self.decoder = Decoder(
            output_channels=output_channels
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.residual_blocks(x)
        x = self.decoder(x)

        return x

In [29]:
# test Generator
generator = Generator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9,
)

sample_input = torch.randn(1, 3, 256, 256)
sample_output = generator(sample_input)

print(f"Generator input shape : {sample_input.shape}")
print(f"Generator output shape: {sample_output.shape}")

Generator input shape : torch.Size([1, 3, 256, 256])
Generator output shape: torch.Size([1, 3, 256, 256])


# Discriminator

## Build PatchGAN Discriminator

In [30]:
class Discriminator(nn.Module):
    def __init__(self, input_channels=3):
        super().__init__()

        self.model = nn.Sequential(
            # Block 1
            nn.Conv2d(
                in_channels=input_channels,
                out_channels=64,
                kernel_size=4,
                stride=2,
                padding=1,
            ),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 2
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 3
            nn.Conv2d(
                in_channels=128,
                out_channels=256,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 4
            nn.Conv2d(
                in_channels=256,
                out_channels=512,
                kernel_size=4,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            # Final PatchGAN prediction layer
            nn.Conv2d(
                in_channels=512,
                out_channels=1,
                kernel_size=4,
                stride=1,
                padding=1,
            ),
        )

    def forward(self, x):
        return self.model(x)

In [31]:
# test patchGAN
discriminator = Discriminator(input_channels=3)

sample_input = torch.randn(1, 3, 256, 256)
patch_output = discriminator(sample_input)

print(f"Discriminator input shape : {sample_input.shape}")
print(f"PatchGAN output shape      : {patch_output.shape}")

Discriminator input shape : torch.Size([1, 3, 256, 256])
PatchGAN output shape      : torch.Size([1, 1, 30, 30])


# Initialize CycleGAN Networks

In [32]:
# Select device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Generators
generator_day_to_night = Generator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9,
).to(device)

generator_night_to_day = Generator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9,
).to(device)

# Discriminators
discriminator_day = Discriminator(
    input_channels=3
).to(device)

discriminator_night = Discriminator(
    input_channels=3
).to(device)

print(f"Device: {device}")
print("CycleGAN networks created successfully.")

Device: cpu
CycleGAN networks created successfully.


In [33]:
sample_day = torch.randn(
    1, 3, 256, 256,
    device=device,
)

sample_night = torch.randn(
    1, 3, 256, 256,
    device=device,
)

with torch.no_grad():
    fake_night = generator_day_to_night(sample_day)
    fake_day = generator_night_to_day(sample_night)

    night_prediction = discriminator_night(fake_night)
    day_prediction = discriminator_day(fake_day)

print(f"Fake night shape       : {fake_night.shape}")
print(f"Fake day shape         : {fake_day.shape}")
print(f"Night patch map shape  : {night_prediction.shape}")
print(f"Day patch map shape    : {day_prediction.shape}")

Fake night shape       : torch.Size([1, 3, 256, 256])
Fake day shape         : torch.Size([1, 3, 256, 256])
Night patch map shape  : torch.Size([1, 1, 30, 30])
Day patch map shape    : torch.Size([1, 1, 30, 30])


# Model Parameter Summary

In [34]:
def count_parameters(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    return total, trainable


models = {
    "Generator Day → Night": generator_day_to_night,
    "Generator Night → Day": generator_night_to_day,
    "Discriminator Day": discriminator_day,
    "Discriminator Night": discriminator_night,
}

for model_name, model in models.items():
    total, trainable = count_parameters(model)

    print(model_name)
    print(f"  Total parameters    : {total:,}")
    print(f"  Trainable parameters: {trainable:,}")

Generator Day → Night
  Total parameters    : 11,372,931
  Trainable parameters: 11,372,931
Generator Night → Day
  Total parameters    : 11,372,931
  Trainable parameters: 11,372,931
Discriminator Day
  Total parameters    : 2,763,841
  Trainable parameters: 2,763,841
Discriminator Night
  Total parameters    : 2,763,841
  Trainable parameters: 2,763,841


# Test Networks with a Real Batch

In [35]:
real_batch = next(iter(train_loader))

real_day = real_batch["day"].to(device)
real_night = real_batch["night"].to(device)

with torch.no_grad():
    fake_night = generator_day_to_night(real_day)
    fake_day = generator_night_to_day(real_night)

    night_patch_map = discriminator_night(fake_night)
    day_patch_map = discriminator_day(fake_day)

print(f"Real day shape       : {real_day.shape}")
print(f"Fake night shape     : {fake_night.shape}")
print(f"Night patch map      : {night_patch_map.shape}")

print(f"Real night shape     : {real_night.shape}")
print(f"Fake day shape       : {fake_day.shape}")
print(f"Day patch map        : {day_patch_map.shape}")

Real day shape       : torch.Size([8, 3, 256, 256])
Fake night shape     : torch.Size([8, 3, 256, 256])
Night patch map      : torch.Size([8, 1, 30, 30])
Real night shape     : torch.Size([8, 3, 256, 256])
Fake day shape       : torch.Size([8, 3, 256, 256])
Day patch map        : torch.Size([8, 1, 30, 30])


# Conclusion

In this notebook, we implemented and validated the complete CycleGAN architecture for unpaired day-to-night road scene translation.

## Implemented Components

- A reusable residual block with reflection padding, convolution, instance normalization, ReLU activation, and a skip connection
- An encoder that converts RGB images from `3 × 256 × 256` into compressed feature representations of `256 × 64 × 64`
- A decoder that reconstructs the compressed representations into RGB images of `3 × 256 × 256`
- A ResNet-based generator containing nine residual blocks
- A PatchGAN discriminator that produces a `30 × 30` patch-level prediction map
- Two generators for day-to-night and night-to-day translation
- Two discriminators for the daytime and nighttime domains

## Architecture Flow

```text
CycleGAN
│
├── Generator: Day → Night
│   ├── Encoder
│   ├── 9 Residual Blocks
│   └── Decoder
│
├── Generator: Night → Day
│   ├── Encoder
│   ├── 9 Residual Blocks
│   └── Decoder
│
├── Day PatchGAN Discriminator
└── Night PatchGAN Discriminator
```

## Validation

The architecture was tested using both synthetic tensors and real batches from the BDD100K data pipeline.

- Generator input and output shapes remained `3 × 256 × 256`
- Encoder output shape was `256 × 64 × 64`
- Decoder restored the original image dimensions
- PatchGAN discriminators produced `30 × 30` prediction maps
- All four CycleGAN networks completed successful forward passes

The models are structurally ready for weight initialization and training, which will be implemented in the next notebook.